# P02. Данные и метрики / Data and metrics

**Colab CPU, 90 минут / minutes.** Сохраните свою копию. Предварительно: P01/HW01 NumPy. Данные искусственные, созданы для занятия, не являются реальными письмами или оценкой реального сервиса.

Save a copy. Prerequisite: P01/HW01 NumPy. The data are synthetic teaching examples, not real messages or an estimate of a real service's performance.

Порядок: 0–10 данные, 10–30 матрица, 30–45 метрики, 45–55 дисбаланс, 55–70 порог и выбор, 70–85 разбиение, 85–90 повторный запуск.

Sequence: data (10 min), matrix (20), metrics (15), imbalance (10), thresholds/selection (15), split (15), clean rerun (5). Practice is ungraded preparation for HW02.


In [ ]:
import sys
import numpy as np
import sklearn
from sklearn.metrics import confusion_matrix, accuracy_score, precision_score, recall_score, f1_score
from sklearn.model_selection import train_test_split
print("Python", sys.version.split()[0], "NumPy", np.__version__, "sklearn", sklearn.__version__)


## 1. Учебные предсказания / Teaching predictions

`0` — обычное сообщение / ordinary message; `1` — спам / spam.
Массивы выровнены по одному порядку объектов. / Arrays share the same example order.


In [ ]:
y = np.r_[np.zeros(80, dtype=int), np.ones(20, dtype=int)]
pred_a = np.r_[np.ones(4, dtype=int), np.zeros(76, dtype=int), np.ones(12, dtype=int), np.zeros(8, dtype=int)]
pred_b = np.r_[np.ones(12, dtype=int), np.zeros(68, dtype=int), np.ones(18, dtype=int), np.zeros(2, dtype=int)]
baseline = np.zeros_like(y)
assert y.shape == pred_a.shape == pred_b.shape == (100,)
print("Class counts:", np.bincount(y))


## 2. Матрица через маски / Matrix from masks

Строки — истинные классы, столбцы — предсказания, порядок `[0,1]`. / Rows are true classes, columns predictions, ordered `[0,1]`.


In [ ]:
tn = np.sum((y == 0) & (pred_a == 0))
fp = np.sum((y == 0) & (pred_a == 1))
fn = np.sum((y == 1) & (pred_a == 0))
tp = np.sum((y == 1) & (pred_a == 1))
cm = np.array([[tn, fp], [fn, tp]])
print(cm)
assert cm.sum() == len(y)
assert np.array_equal(cm, confusion_matrix(y, pred_a, labels=[0,1]))


**Упражнение / Exercise:** повторите для B. Ожидается `[[68,12],[2,18]]`. Объясните смысл ячейки 12. / Repeat for B and explain the cell containing 12.

In [ ]:
# TODO: matrix for B

## 3. Метрики / Metrics

Для нулевого знаменателя используем 0. / Use 0 for a zero denominator.


In [ ]:
def safe_ratio(a, b):
    return float(a / b) if b else 0.0

manual = dict(accuracy=safe_ratio(tp+tn, len(y)),
              precision=safe_ratio(tp, tp+fp),
              recall=safe_ratio(tp, tp+fn),
              f1=safe_ratio(2*tp, 2*tp+fp+fn))
reference = dict(accuracy=accuracy_score(y,pred_a),
                 precision=precision_score(y,pred_a,zero_division=0),
                 recall=recall_score(y,pred_a,zero_division=0),
                 f1=f1_score(y,pred_a,zero_division=0))
print(manual)
assert all(np.isclose(manual[k], reference[k]) for k in manual)


**Упражнение / Exercise:** вычислите те же метрики для B. Почему accuracy ниже, а F1 выше? / Compute B's metrics. Why is accuracy lower while F1 is higher?

In [ ]:
# TODO

## 4. Дисбаланс / Imbalance

Это отдельный пример с 99% нулей. / This is a separate example with 99% zeros.


In [ ]:
rare_y = np.r_[np.zeros(990,dtype=int),np.ones(10,dtype=int)]
never_spam = np.zeros_like(rare_y)
print("Accuracy:", accuracy_score(rare_y,never_spam))
print("Recall:", recall_score(rare_y,never_spam,zero_division=0))


**Обсудите / Discuss:** какое полезное действие не выполняет такой baseline? / Which useful action does this baseline fail to perform?

## 5. Сравнение и ограничение / Comparison and constraint

Правило до просмотра результатов: максимум recall при precision ≥ 0.70. / Predetermined rule: maximise recall subject to precision ≥ 0.70.


In [ ]:
reports = {}
for name, pred in {"Baseline":baseline,"A":pred_a,"B":pred_b}.items():
    reports[name] = {"accuracy":accuracy_score(y,pred),
                     "precision":precision_score(y,pred,zero_division=0),
                     "recall":recall_score(y,pred,zero_division=0),
                     "f1":f1_score(y,pred,zero_division=0)}
    print(name, reports[name])
eligible = [name for name in reports if reports[name]["precision"] >= .70]
print("Eligible:",eligible)


**Упражнение / Exercise:** назовите выбранный вариант и объясните, почему максимальный F1 не определяет ответ. / Select a candidate and explain why the highest F1 does not determine the answer.

## 6. Порог / Threshold

Независимый маленький пример: scores — условные оценки, не калиброванные вероятности. / Separate small example: scores are illustrative, not calibrated probabilities.


In [ ]:
threshold_y = np.array([0,0,0,0,1,1,1,1])
scores = np.array([.1,.2,.4,.8,.3,.6,.7,.9])
for threshold in [.3,.5,.7,.9]:
    pred = (scores >= threshold).astype(int)
    print(threshold, "P=", precision_score(threshold_y,pred,zero_division=0),
          "R=", recall_score(threshold_y,pred,zero_division=0))


**Упражнение / Exercise:** выберите порог по тому же условию precision ≥ 0.70. Что изменится при `>` вместо `>=`? / Select a threshold using the same constraint. What changes if `>` replaces `>=`?

In [ ]:
# TODO

## 7. Разбиение 60/20/20 / A 60/20/20 split

Предполагаем независимые строки. При группах/времени нужна другая схема. / We assume independent rows; groups or time dependence require a different scheme.


In [ ]:
ids = np.arange(1000)
labels = np.r_[np.zeros(800,dtype=int),np.ones(200,dtype=int)]
trainval_ids, test_ids = train_test_split(ids,test_size=.2,random_state=42,stratify=labels)
train_ids, val_ids = train_test_split(trainval_ids,test_size=.25,random_state=42,
                                     stratify=labels[trainval_ids])
assert len(train_ids)==600 and len(val_ids)==200 and len(test_ids)==200
assert not (set(train_ids)&set(val_ids) or set(train_ids)&set(test_ids) or set(val_ids)&set(test_ids))
for name,part in [("Train",train_ids),("Validation",val_ids),("Test",test_ids)]:
    print(name,len(part),np.bincount(labels[part]))
experiment_log = {"dataset":"synthetic_independent_rows_v1","seed":42,
                  "numpy":np.__version__,"sklearn":sklearn.__version__,
                  "selection":"max recall subject to precision >= .70"}
print(experiment_log)


**Обсудите / Discuss:** почему фиксация seed не доказывает репрезентативность? Почему нельзя нормализовать по полному набору до разбиения? / Why does fixing the seed not prove representativeness? Why should preprocessing not be fitted on the full dataset before splitting?

## Дальше / Next

[HW02 в Colab / HW02 in Colab](https://colab.research.google.com/github/alexander-toschev/ml-cs-intro/blob/main/home-work/HW02_Data_Metrics.ipynb). Перезапустите среду и выполните все ячейки. / Restart and run all cells.
